In [ ]:
import numpy as np
import gym
from dezero import Model
from dezero import optimizers
import dezero.functions as F
import dezero.layers as L

class PolicyNet(Model): # 정책 신경망
    def __init__(self, action_size=2):
        super().__init__()
        self.l1 = L.Linear(64)
        self.l2 = L.Linear(action_size)

    def forward(self,x):
        x = F.relu(self.l1(x))
        x = self.l2(x)
        x = F.softmax(x)    # 확률 출력
        return x
    
class ValueNet(Model): # 가치 함수 신경망
    def __init__(self):
        super().__init__()
        self.l1 = L.Linear(64)
        self.l2 = L.Linear(1)

    def forward(self,x):
        x = F.relu(self.l1(x)) 
        x = self.l2(x)

        return x
    
class Agent:
    def __init__(self):
        # ! original Hp
        # self.gamma = 0.98
        # self.lr_pi = 0.0002
        # self.lr_v = 0.0005
        
        # ! modified -1 
        self.gamma = 0.99
        self.lr_pi = 1e-4
        self.lr_v = 1e-3
        
        
        self.action_size = 3

        self.pi = PolicyNet(action_size=self.action_size)
        self.v = ValueNet()
        self.optimizer_pi = optimizers.Adam(self.lr_pi).setup(self.pi)
        self.optimizer_v = optimizers.Adam(self.lr_v).setup(self.v)

    def get_action(self, state):
        state = state[np.newaxis, :]     # 배치 처리용 축 추가
        probs = self.pi(state)
        probs = probs[0]
        action = np.random.choice(len(probs), p=probs.data)
        return action, probs[action] # 선택된 행동과 해당 행동의 확률 반환
    
    def update(self, state, action_prob, reward, next_state, done):
        # 배치 처리용 축 추가
        state = state[np.newaxis, :]
        next_state = next_state[np.newaxis, :]

        # 가치 함수의 손실 계산
        # print(f"{state}, {abs(state[:,1] * 10)}")
        # print(f"{state[:,1]}")
        target = reward + self.gamma * (self.v(next_state) + abs(state[:,1] * 100000)) * (1- done) # TD 목표
        target.unchain()
        v = self.v(state) # 현재 상태의 가치 함수
        loss_v = F.mean_squared_error(v, target) # 두 값의 평균제곱오차

        # 정책의 손실 계산
        delta = target -v
        delta.unchain()
        loss_pi = -F.log(action_prob) * delta
        
        # print(f"{state} {action_prob} {v} {loss_v.data}")

        self.v.cleargrads()
        self.pi.cleargrads()
        loss_v.backward()
        loss_pi.backward()
        self.optimizer_v.update()
        self.optimizer_pi.update()




episodes = 3000
env = gym.make('MountainCar-v0', render_mode='rgb_array')
# env = gym.make('MountainCar-v0', render_mode='human')
agent = Agent()
reward_history = []

for episode in range(episodes):
    state = env.reset()[0]
    done = False
    total_reward = 0

    while not done:
        action, prob = agent.get_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        # print(f"{next_state}, {reward}, {info}")
        
        done= terminated | truncated

        agent.update(state, prob, reward, next_state, done)

        state = next_state
        total_reward += reward

    reward_history.append(total_reward)
    if episode % 100 == 0:
        # env.render()
        print("episode: {}, total reward: {:.1f}".format(episode, total_reward))
    # print(f"# episode: {episode}")
    # print("episode: {}, total reward: {:.1f}".format(episode, total_reward))

# 그래프
from common.utils import plot_total_reward
# plot_total_reward(reward_history)





# 학습 끝난 에이전트에 탐욕 행동을 선택하도록 하여 플레이
env2 = gym.make("MountainCar-v0", render_mode='human')

state = env2.reset()[0]
done = False
total_reward = 0

while not done:
    action, prob = agent.get_action(state)
    next_state, reward, terminated, truncated, info = env2.step(action)
    done = terminated | truncated

    agent.update(state, prob, reward, next_state, done)

    state = next_state
    total_reward += reward
    env2.render()

print("total reward: {}".format(total_reward))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[0.]
[-0.00043841]
[0.00012637]
[0.00069024]
[0.00124906]
[0.00079875]
[0.00034259]
[0.00088392]
[0.00141878]
[0.00194324]
[0.00345344]
[0.00493826]
[0.00538669]
[0.00579532]
[0.00616101]
[0.00748093]
[0.00874513]
[0.00894402]
[0.00907598]
[0.00813993]
[0.00914283]
[0.00807718]
[0.00895098]
[0.0077578]
[0.00650666]
[0.00620702]
[0.00686122]
[0.00746453]
[0.00701267]
[0.00550919]
[0.0049653]
[0.00538513]
[0.00476575]
[0.0041118]
[0.00442814]
[0.0047126]
[0.00496326]
[0.0041785]
[0.00436404]
[0.00251871]
[0.00065562]
[-0.00121208]
[-0.00307124]
[-0.00390874]
[-0.00371856]
[-0.00550195]
[-0.00724604]
[-0.00793808]
[-0.00857272]
[-0.00814497]
[-0.00865759]
[-0.01010649]
[-0.01048063]
[-0.01077687]
[-0.01199272]
[-0.01311884]
[-0.01414662]
[-0.01506831]
[-0.01587717]
[-0.01556756]
[-0.01514247]
[-0.01560594]
[-0.01495572]
[-0.01419789]
[-0.0133393]
[-0.01138746]
[-0.01035722]
[-0.00925671]
[-0.0080943]
[-0.00687849]
[-0.00461787]
[-0.00232746]
[-0.00102213]
[0.00128974]
[0.00359335]
[0.0048

KeyboardInterrupt: 